# Thesis Pipeline Orchestrator
This notebook starts the Neo4j environment, optionally rebuilds the dataset, selects an optimized historical/detection window,
and runs the full label-aware and label-agnostic FastRP analyses. Artifacts are archived under `thesis_results/` for thesis use.

In [ ]:
# Global configuration for the thesis analysis pipeline
from pathlib import Path
from datetime import datetime
import glob
import json
import shutil
import pandas as pd
import numpy as np

from CART import Controller

# Window configuration (optimized from prior sweep)
HISTORICAL_WINDOW_HOURS = 48
DETECTION_WINDOW_HOURS = 24
EMBEDDING_DIM = 128

# Multi-hop chain analysis configuration - NEW INTEGRATED APPROACH
# Now uses CART.analyzers.SubnetPivotAnalyzer.analyze_multi_hop_chains()
# with intelligent caching and configurable n-hops
N_HOPS = 3  # Default: 3-hop chains (produces 4-node chains A→B→C→D)
USE_CHAIN_CACHE = True  # Enable caching for faster re-runs

# Pipeline control flags
REBUILD_DATABASE = True    # Set True only if you need fresh data from source
RUN_WINDOW_SWEEP = False    # EXPENSIVE - Already optimized, keep False
USE_OPTIMIZED_WINDOW = True # Use pre-computed optimal windows (48h/24h)

# Experiment selection
RUN_LABEL_AWARE = True      # Run MITRE ATT&CK-aware experiment
RUN_LABEL_AGNOSTIC = True   # Run structural (label-agnostic) experiment
LABEL_AGNOSTIC_LIMIT = None # Optional cap for label-agnostic recon events

# Post-processing
SHUTDOWN_AFTER_RUN = False        # Stop container when notebook completes
GENERATE_THESIS_ARTIFACTS = True  # Run artifact generator after pipeline completes

OUTPUT_DIR = Path('thesis_results')

In [ ]:
# Start or connect to the shared Neo4j controller container
controller = Controller()

In [ ]:

status = controller.status()
if not status.get('running'):
    print('Starting Neo4j container...')
    controller.start()
else:
    print('Neo4j container already running.')

if not controller.connect():
    raise RuntimeError('Could not connect to Neo4j; check container logs.')
print('Controller connected to Neo4j.')

In [ ]:
# Optional database rebuild (downloads the dataset and reimports into Neo4j)
if REBUILD_DATABASE:
    print('Rebuilding database with full dataset...')
    controller.build_database(rebuild=True)
else:
    print('Skipping database rebuild; set REBUILD_DATABASE=True to rebuild.')

In [ ]:
# Optional window sweep to regenerate comparison CSVs
if RUN_WINDOW_SWEEP:
    import optimize_windows
    optimize_windows.input = lambda prompt='': None
    optimize_windows.main()
else:
    print('Skipping window sweep; set RUN_WINDOW_SWEEP=True to execute optimize_windows.py.')

In [ ]:
# Evaluate available window optimization outputs and optionally update window selection
records = []
for path in sorted(glob.glob('window_opt_*_method_comparison.csv')):
    parts = Path(path).stem.split('_')
    hist = int(parts[2])
    det = int(parts[3])
    df = pd.read_csv(path)
    if 'FastRP Embedding' not in df['Method'].values:
        continue
    row = df[df['Method'] == 'FastRP Embedding'].iloc[0]
    pred_path = path.replace('_method_comparison.csv', '_pivot_predictions.csv')
    try:
        pred_df = pd.read_csv(pred_path, usecols=['became_pivot'])
        pivot_rate = pred_df['became_pivot'].mean() * 100
        sample_count = len(pred_df)
        pivot_count = pred_df['became_pivot'].sum()
    except Exception:
        pivot_rate = np.nan
        sample_count = np.nan
        pivot_count = np.nan
    records.append({
        'historical_hours': hist,
        'detection_hours': det,
        'auc_roc': row['AUC-ROC'],
        'auc_pr': row['AUC-PR'],
        'f1': row['F1-Score'],
        'pivot_rate': pivot_rate,
        'samples': sample_count,
        'pivot_count': pivot_count
    })

if records:
    window_df = pd.DataFrame(records).sort_values(
        ['auc_roc', 'historical_hours', 'detection_hours'],
        ascending=[False, True, True]
    )
    print(window_df.to_string(index=False))
    if USE_OPTIMIZED_WINDOW:
        best_row = window_df.iloc[0]
        HISTORICAL_WINDOW_HOURS = int(best_row['historical_hours'])
        DETECTION_WINDOW_HOURS = int(best_row['detection_hours'])
        print(f"Using best window: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h")
else:
    print('No window optimization files found; retaining configured windows.')

print(f'Analysis window configuration: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h')

In [ ]:
# Prepare SubnetPivotAnalyzer with full-corpus reconnaissance sampling
analyzer = controller.SubnetPivotAnalyzer
if not analyzer.connect():
    raise RuntimeError('Analyzer could not connect to Neo4j.')

def patch_full_recon_sampling(analyzer_obj, label_agnostic_limit=None):
    original_fn = analyzer_obj.identify_reconnaissance_victims_by_subnet

    def patched(self, use_labels: bool, historical_window_hours: int):
        print('\n--- Identifying Reconnaissance Victims by Subnet (full corpus) ---')
        with self.driver.session(database=self.database) as session:
            if use_labels:
                query = (
                    "MATCH (a:IP)-[r:CONNECTS]->(v:IP)\n"
                    "WHERE r.is_attack = 1 AND r.tactic = 'Reconnaissance'\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
            else:
                query = (
                    "MATCH (a:IP)-[r1:CONNECTS]->(v:IP)\n"
                    "WHERE exists { (v)-[:CONNECTS]->() }\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r1.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
                if label_agnostic_limit is not None:
                    query += f'\nLIMIT {int(label_agnostic_limit)}'
            result = session.run(query).data()
        print(f"  ✓ Found {len(result):,} reconnaissance events")
        return result

    analyzer_obj.identify_reconnaissance_victims_by_subnet = patched.__get__(analyzer_obj, analyzer_obj.__class__)
    return original_fn

original_recon_fn = patch_full_recon_sampling(analyzer, label_agnostic_limit=LABEL_AGNOSTIC_LIMIT)

In [ ]:
# Run selected experiments with the configured window
run_modes = []
if RUN_LABEL_AWARE:
    run_modes.append('label_aware')
if RUN_LABEL_AGNOSTIC:
    run_modes.append('label_agnostic')

if not run_modes:
    raise ValueError('No experiments selected; enable RUN_LABEL_AWARE and/or RUN_LABEL_AGNOSTIC.')

def mode_artifacts_available(prefix: str) -> bool:
    pivot_path = Path(f"{prefix}_pivot_predictions.csv")
    method_path = Path(f"{prefix}_method_comparison.csv")
    missing = [p.name for p in (pivot_path, method_path) if not p.exists()]
    if missing:
        print(f"  ⚠ Missing artifacts for {prefix}: {', '.join(missing)}")
        return False
    return True

executed_prefixes = []
overall_start = datetime.utcnow()
try:
    for mode in run_modes:
        print('\n' + '=' * 80)
        print(f"Executing {mode.replace('_', ' ').title()} experiment")
        print('=' * 80)
        mode_start = datetime.utcnow()
        analyzer.run_full_analysis(
            mode=mode,
            historical_window_hours=HISTORICAL_WINDOW_HOURS,
            detection_window_hours=DETECTION_WINDOW_HOURS,
            embedding_dim=EMBEDDING_DIM,
            n_hops=N_HOPS  # Use new integrated n-hop chain analysis
        )
        mode_finish = datetime.utcnow()
        if mode_artifacts_available(mode):
            executed_prefixes.append(mode)
        else:
            print(f"  ⚠ Skipping downstream steps for {mode}; required artifacts not produced.")
        print(f"{mode} runtime: {(mode_finish - mode_start).total_seconds():.1f} seconds")

    if {'label_aware', 'label_agnostic'}.issubset(set(executed_prefixes)):
        analyzer.compare_analysis_modes()
    else:
        print("  Comparison skipped; ensure both modes complete successfully before comparing.")
finally:
    analyzer.identify_reconnaissance_victims_by_subnet = original_recon_fn
    overall_finish = datetime.utcnow()
    print(f"Total analysis runtime: {(overall_finish - overall_start).total_seconds():.1f} seconds")

In [ ]:
# Collect and archive artifacts for this run
run_stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
run_dir = OUTPUT_DIR / f'run_{run_stamp}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}'
run_dir.mkdir(parents=True, exist_ok=True)

def add_window_tag(name: str) -> str:
    if name.startswith('label_aware_'):
        return name.replace('label_aware_', f"label_aware_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    if name.startswith('label_agnostic_'):
        return name.replace('label_agnostic_', f"label_agnostic_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    return name

patterns = [f"{prefix}_*" for prefix in executed_prefixes]
patterns.append('mode_comparison.png')

moved = []
for pattern in patterns:
    for src_path in Path('.').glob(pattern):
        if not src_path.is_file():
            continue
        dest_name = add_window_tag(src_path.name)
        dest_path = run_dir / dest_name
        shutil.move(str(src_path), dest_path)
        moved.append(dest_path)
        print(f'Moved {src_path.name} -> {dest_path}')

metadata = {
    'timestamp_utc': run_stamp,
    'historical_window_hours': HISTORICAL_WINDOW_HOURS,
    'detection_window_hours': DETECTION_WINDOW_HOURS,
    'embedding_dim': EMBEDDING_DIM,
    'executed_prefixes': executed_prefixes,
    'artifacts': [str(path.name) for path in moved]
}
(run_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Archived run artifacts in {run_dir}')

In [ ]:
# Summarize key metrics from the archived results
def load_artifact(run_directory: Path, prefix: str, suffix: str) -> Path | None:
    pattern = f"{prefix}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_{suffix}"
    matches = list(run_directory.glob(pattern))
    return matches[0] if matches else None

aware_method_path = load_artifact(run_dir, 'label_aware', 'method_comparison.csv') if 'label_aware' in executed_prefixes else None
agnostic_method_path = load_artifact(run_dir, 'label_agnostic', 'method_comparison.csv') if 'label_agnostic' in executed_prefixes else None
aware_preds_path = load_artifact(run_dir, 'label_aware', 'pivot_predictions.csv') if 'label_aware' in executed_prefixes else None
agnostic_preds_path = load_artifact(run_dir, 'label_agnostic', 'pivot_predictions.csv') if 'label_agnostic' in executed_prefixes else None

summary_rows = []
for label, method_path, preds_path in [
    ('Label-Aware', aware_method_path, aware_preds_path),
    ('Label-Agnostic', agnostic_method_path, agnostic_preds_path)
]:
    if method_path is None or preds_path is None:
        print(f'Missing artifacts for {label}; skip summary.')
        continue
    method_df = pd.read_csv(method_path)
    preds_df = pd.read_csv(preds_path)
    if 'FastRP Embedding' not in method_df['Method'].values:
        print(f'FastRP results missing for {label}; skip summary.')
        continue
    fastrp_row = method_df[method_df['Method'] == 'FastRP Embedding'].iloc[0]
    pivot_rate = preds_df['became_pivot'].mean() * 100 if 'became_pivot' in preds_df.columns else np.nan
    summary_rows.append({
        'Mode': label,
        'Samples': len(preds_df),
        'Pivots': preds_df['became_pivot'].sum() if 'became_pivot' in preds_df.columns else np.nan,
        'Pivot Rate (%)': pivot_rate,
        'AUC-ROC': fastrp_row['AUC-ROC'],
        'AUC-PR': fastrp_row['AUC-PR'],
        'F1-Score': fastrp_row['F1-Score'],
        'Precision': fastrp_row['Precision'],
        'Recall': fastrp_row['Recall']
    })

def to_builtin(value):
    if isinstance(value, (np.generic,)):
        return value.item()
    return value

if summary_rows:
    summary_rows_builtin = [
        {key: to_builtin(value) for key, value in row.items()}
        for row in summary_rows
    ]
    summary_df = pd.DataFrame(summary_rows_builtin)
    print(summary_df.to_string(index=False))
    summary_payload = metadata.copy()
    summary_payload['metrics'] = summary_rows_builtin
    (run_dir / 'run_summary.json').write_text(json.dumps(summary_payload, indent=2))
else:
    print('No summary generated; verify artifacts above.')

In [ ]:
# Export CONNECTS edges for visualization artifacts
from CART.base import Neo4jConnection

CONNECTS_EXPORT_PATH = OUTPUT_DIR / 'connects_edges.csv'
export_cypher = """
CALL apoc.export.csv.query(
  "MATCH (a:IP)-[r:CONNECTS]->(b:IP)\n   RETURN a.address AS src,\n          b.address AS dst,\n          r.timestamp AS ts,\n          r.is_attack AS is_attack",
  'thesis_results/connects_edges.csv',
  {batchSize: 50000, delimiter: ',', quotes: false}
 )
YIELD file, source, format, nodes, relationships, properties, time
RETURN file, source, format, nodes, relationships, properties, time;
"""

needs_export = REFRESH_CONNECTS_EXPORT or not CONNECTS_EXPORT_PATH.exists()
if needs_export:
    print('Exporting IP→IP CONNECTS edges via APOC...')
    if not analyzer.connect():
        raise RuntimeError('SubnetPivotAnalyzer could not reconnect for export; check Neo4j status.')
    with analyzer.driver.session(database=analyzer.database) as session:
        check_query = (
            "SHOW PROCEDURES YIELD name "
            "WHERE name = 'apoc.export.csv.query' "
            "RETURN count(*) AS matches"
        )
        matches = session.run(check_query).single()["matches"]
        if matches == 0:
            raise RuntimeError("apoc.export.csv.query is not registered. Restart the Neo4j container with APOC enabled.")
        summary_records = session.run(export_cypher).data()
        if not summary_records:
            raise RuntimeError('APOC export returned no metadata; inspect Neo4j logs for details.')
        print('CSV export complete:')
        for key, value in summary_records[0].items():
            print(f'  {key}: {value}')
else:
    print(f'Using existing CONNECTS export at {CONNECTS_EXPORT_PATH}')

Neo4jConnection().ensure_export_permissions()
connects_df = pd.read_csv(CONNECTS_EXPORT_PATH, usecols=['src', 'dst', 'ts', 'is_attack'])
connects_df['ts'] = connects_df['ts'].astype('int64')
print(f'Loaded {len(connects_df):,} edges for downstream visualization.')

In [ ]:
# Build visualizations using the chain results from CART analyzer
# The multi-hop chain analysis is now integrated into run_full_analysis()
# and results are saved automatically by CART.analyzers.SubnetPivotAnalyzer

from typing import Literal, Optional, Dict, List, Tuple
import os

try:
    import networkx as nx
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        "Install networkx, matplotlib, and seaborn before running visualization steps: `pip install networkx matplotlib seaborn`."
    ) from exc

ROLE_COLORS = {
    'attacker': '#e41a1c',
    'pivot': '#ffd60a',
    'victim_only': '#7b1fa2',
    'neutral': '#2ca02c',
}

ROLE_LABELS = {
    'attacker': 'Attacker',
    'pivot': 'Pivot (victim → attacker)',
    'victim_only': 'Victim only',
    'neutral': 'Non-victim/attacker',
}

SUBNET_PREFIX = 24

def _ip_to_subnet(ip: str, prefix: int = SUBNET_PREFIX) -> str:
    if prefix != 24:
        raise ValueError('Only /24 subnets are supported right now.')
    parts = ip.split('.')
    if len(parts) != 4:
        return ip
    return '.'.join(parts[:3]) + '.0/24'

def _classify_roles(
    df: pd.DataFrame,
    *,
    aggregate_by_subnet: bool = False,
    subnet_prefix: int = SUBNET_PREFIX,
 ) -> tuple[set[str], set[str], set[str], set[str]]:
    attack_edges = df[df['is_attack'] == 1]
    attack_sources = set(attack_edges['src'])
    attack_targets = set(attack_edges['dst'])
    pivot_nodes = attack_sources & attack_targets
    attacker_only = attack_sources - pivot_nodes
    victim_only = attack_targets - attack_sources
    all_nodes = set(df['src']).union(set(df['dst']))
    neutral_nodes = all_nodes - attack_sources - attack_targets

    if not aggregate_by_subnet:
        return attacker_only, pivot_nodes, victim_only, neutral_nodes

    def _transform(nodes: set[str]) -> set[str]:
        return {_ip_to_subnet(node, subnet_prefix) for node in nodes}

    return (
        _transform(attacker_only),
        _transform(pivot_nodes),
        _transform(victim_only),
        _transform(neutral_nodes),
    )

def _subset_for_visualization(
    df: pd.DataFrame,
    *,
    include_neutral: bool,
    max_nodes: Optional[int],
    max_edges: Optional[int],
    aggregate_by_subnet: bool,
    subnet_prefix: int,
 ) -> tuple[pd.DataFrame, set[str], dict[str, str]]:
    attacker_only, pivot_nodes, victim_only, neutral_nodes = _classify_roles(
        df, aggregate_by_subnet=aggregate_by_subnet, subnet_prefix=subnet_prefix
    )
    attack_related_nodes = attacker_only | pivot_nodes | victim_only

    working_df = df.copy()
    if not include_neutral:
        working_df = working_df[
            working_df['src'].isin(attack_related_nodes) & working_df['dst'].isin(attack_related_nodes)
        ]

    if working_df.empty:
        return working_df, set(), {}

    working_df = working_df.sort_values(['is_attack', 'ts'], ascending=[False, True])

    if aggregate_by_subnet:
        working_df = working_df.assign(
            src=working_df['src'].map(lambda ip: _ip_to_subnet(ip, subnet_prefix)),
            dst=working_df['dst'].map(lambda ip: _ip_to_subnet(ip, subnet_prefix)),
        )
        grouped = working_df.groupby(['src', 'dst'], as_index=False).agg(
            ts=('ts', 'min'),
            is_attack=('is_attack', 'max'),
            edge_count=('ts', 'count'),
            attack_edges=('is_attack', 'sum'),
        )
        grouped['is_attack'] = grouped['is_attack'].astype(int)
        working_df = grouped.sort_values(['is_attack', 'ts'], ascending=[False, True])

    if max_edges is not None and len(working_df) > max_edges:
        print(
            f'Truncating edges to top {max_edges} of {len(working_df)} rows (prioritised by attack flag, then time).'
        )
        working_df = working_df.head(max_edges)

    degree_counts = (
        pd.concat([working_df['src'], working_df['dst']]).value_counts().rename('degree')
    )
    node_subset: set[str] = set(degree_counts.index)

    if max_nodes is not None and len(node_subset) > max_nodes:
        def _priority(node: str) -> tuple[int, int]:
            if node in pivot_nodes:
                role_rank = 0
            elif node in attacker_only:
                role_rank = 1
            elif node in victim_only:
                role_rank = 2
            else:
                role_rank = 3
            return role_rank, -int(degree_counts.get(node, 0))

        ordered_nodes = sorted(node_subset, key=_priority)
        kept_nodes = ordered_nodes[:max_nodes]
        print(
            f'Truncating nodes to {max_nodes} of {len(node_subset)} (prioritising pivots, attackers, victims, then neutrals by degree).'
        )
        node_subset = set(kept_nodes)
        working_df = working_df[
            working_df['src'].isin(node_subset) & working_df['dst'].isin(node_subset)
        ]

    if working_df.empty:
        return working_df, set(), {}

    node_subset = set(working_df['src']).union(set(working_df['dst']))

    role_map: dict[str, str] = {}
    for node in node_subset:
        if node in pivot_nodes:
            role_map[node] = 'pivot'
        elif node in attacker_only:
            role_map[node] = 'attacker'
        elif node in victim_only:
            role_map[node] = 'victim_only'
        else:
            role_map[node] = 'neutral'

    return working_df, node_subset, role_map

def build_attack_graph(
    df: pd.DataFrame,
    *,
    include_neutral: bool = True,
    max_nodes: Optional[int] = None,
    max_edges: Optional[int] = None,
    aggregate_by_subnet: bool = False,
    subnet_prefix: int = SUBNET_PREFIX,
 ) -> tuple[nx.DiGraph, dict[str, str], pd.DataFrame]:
    edges_df, node_subset, role_map = _subset_for_visualization(
        df,
        include_neutral=include_neutral,
        max_nodes=max_nodes,
        max_edges=max_edges,
        aggregate_by_subnet=aggregate_by_subnet,
        subnet_prefix=subnet_prefix,
    )

    graph = nx.DiGraph()
    for node, role in role_map.items():
        graph.add_node(node, role=role)
    for row in edges_df.itertuples(index=False):
        edge_attrs = {'is_attack': int(getattr(row, 'is_attack', 0))}
        if hasattr(row, 'edge_count'):
            edge_attrs['edge_count'] = getattr(row, 'edge_count')
        if hasattr(row, 'attack_edges'):
            edge_attrs['attack_edges'] = getattr(row, 'attack_edges')
        graph.add_edge(row.src, row.dst, **edge_attrs)

    return graph, role_map, edges_df

def draw_attack_graph(
    graph: nx.DiGraph,
    role_map: dict[str, str],
    *,
    title: str,
    layout: str = 'spring',
    seed: int = 42,
    figsize: tuple[int, int] = (16, 12),
    node_size: int = 60,
    edge_alpha: float = 0.18,
    edge_weight_attr: Optional[str] = None,
    attack_edge_width: float = 1.8,
    benign_edge_width: float = 0.6,
    save_path: Optional[Path] = None,
 ) -> None:
    if graph.number_of_nodes() == 0:
        print('No nodes available to draw; adjust filtering parameters.')
        return

    if layout == 'spring':
        pos = nx.spring_layout(graph, seed=seed)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(graph)
    else:
        pos = nx.random_layout(graph, seed=seed)

    plt.figure(figsize=figsize)
    for role, color in ROLE_COLORS.items():
        nodes = [n for n in graph.nodes if role_map.get(n) == role]
        if not nodes:
            continue
        nx.draw_networkx_nodes(
            graph,
            pos,
            nodelist=nodes,
            node_color=color,
            node_size=node_size,
            alpha=0.85,
            label=ROLE_LABELS[role],
        )

    attack_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('is_attack') == 1]
    benign_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('is_attack') != 1]

    def _edge_widths(edge_list: list[tuple[str, str]], base_width: float) -> list[float]:
        if not edge_list:
            return []
        if edge_weight_attr is None:
            return [base_width] * len(edge_list)
        weights = [float(graph[u][v].get(edge_weight_attr, 1.0)) for u, v in edge_list]
        max_weight = max(weights)
        if max_weight <= 0:
            return [base_width] * len(edge_list)
        min_width = max(base_width * 0.35, 0.35)
        scale = base_width / max_weight
        return [max(weight * scale, min_width) for weight in weights]

    if benign_edges:
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=benign_edges,
            edge_color='#94a3b8',
            alpha=edge_alpha,
            arrows=False,
            width=_edge_widths(benign_edges, benign_edge_width),
        )
    if attack_edges:
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=attack_edges,
            edge_color='#ff595e',
            alpha=max(edge_alpha, 0.3),
            arrows=False,
            width=_edge_widths(attack_edges, attack_edge_width),
        )

    plt.title(title)
    plt.axis('off')
    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label=ROLE_LABELS[role],
               markerfacecolor=color, markersize=10)
        for role, color in ROLE_COLORS.items()
    ]
    plt.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

chain_prefix = f"h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}"

print(f'\n{"="*80}')
print(f'BUILDING NETWORK VISUALIZATIONS')
print(f'{"="*80}')
print(f'Note: Multi-hop chain analysis is now integrated into CART analyzer')
print(f'Chain results saved as: label_aware_{N_HOPS}hop_chains.csv')
print(f'                        label_agnostic_{N_HOPS}hop_chains.csv')
print(f'{"="*80}\n')

# Load CONNECTS edges for network graph visualization
CONNECTS_EXPORT_PATH = OUTPUT_DIR / 'connects_edges.csv'
if not CONNECTS_EXPORT_PATH.exists():
    print(f'⚠ CONNECTS export not found at {CONNECTS_EXPORT_PATH}')
    print('  Skipping network visualizations')
else:
    connects_df = pd.read_csv(CONNECTS_EXPORT_PATH, usecols=['src', 'dst', 'ts', 'is_attack'])
    connects_df['ts'] = connects_df['ts'].astype('int64')
    print(f'Loaded {len(connects_df):,} edges for network visualization.')

    # Build full network graphs
    label_aware_df = connects_df[connects_df['is_attack'] == 1].copy()
    if label_aware_df.empty:
        print('Label-aware dataset produced no attack edges; skipping network visualizations.')
    else:
        la_graph, la_role_map, la_edges = build_attack_graph(
            label_aware_df,
            include_neutral=True,
            max_nodes=None,
            max_edges=None,
        )
        print(
            f"Label-aware IP graph: {la_graph.number_of_nodes():,} nodes, {la_graph.number_of_edges():,} edges "
            f"(attack edges = {(la_edges['is_attack'] == 1).sum():,})."
        )
        draw_attack_graph(
            la_graph,
            la_role_map,
            title='Label-aware attack flow (IP-level)',
            layout='spring',
            seed=24,
            figsize=(20, 16),
            node_size=35,
            edge_alpha=0.12,
            save_path=run_dir / f'label_aware_{chain_prefix}_ip_graph.png',
        )

        la_subnet_graph, la_subnet_role_map, la_subnet_edges = build_attack_graph(
            label_aware_df,
            include_neutral=True,
            max_nodes=None,
            max_edges=None,
            aggregate_by_subnet=True,
            subnet_prefix=SUBNET_PREFIX,
        )
        total_la_edges = (
            int(la_subnet_edges['edge_count'].sum())
            if 'edge_count' in la_subnet_edges.columns
            else len(la_subnet_edges)
        )
        print(
            f"Label-aware subnet graph: {la_subnet_graph.number_of_nodes():,} subnets, {la_subnet_graph.number_of_edges():,} edges "
            f"(underlying edges represented = {total_la_edges:,})."
        )
        draw_attack_graph(
            la_subnet_graph,
            la_subnet_role_map,
            title='Label-aware attack flow (subnet /24)',
            layout='spring',
            seed=32,
            figsize=(18, 14),
            node_size=140,
            edge_alpha=0.25,
            edge_weight_attr='edge_count',
            attack_edge_width=2.6,
            benign_edge_width=1.2,
            save_path=run_dir / f'label_aware_{chain_prefix}_subnet_graph.png',
        )

    full_graph, full_role_map, full_edges = build_attack_graph(
        connects_df,
        include_neutral=True,
        max_nodes=None,
        max_edges=None,
     )
    print(
        f"Label-agnostic IP graph: {full_graph.number_of_nodes():,} nodes, {full_graph.number_of_edges():,} edges "
        f"(attack edges = {(full_edges['is_attack'] == 1).sum():,})."
     )
    draw_attack_graph(
        full_graph,
        full_role_map,
        title='Label-agnostic attack flow (IP-level)',
        layout='spring',
        seed=20,
        figsize=(20, 16),
        node_size=32,
        edge_alpha=0.1,
        save_path=run_dir / f'label_agnostic_{chain_prefix}_ip_graph.png',
     )

    subnet_graph, subnet_role_map, subnet_edges = build_attack_graph(
        connects_df,
        include_neutral=True,
        max_nodes=None,
        max_edges=None,
        aggregate_by_subnet=True,
        subnet_prefix=SUBNET_PREFIX,
     )
    total_edges = (
        int(subnet_edges['edge_count'].sum()) if 'edge_count' in subnet_edges.columns else len(subnet_edges)
     )
    print(
        f"Label-agnostic subnet graph: {subnet_graph.number_of_nodes():,} subnets, {subnet_graph.number_of_edges():,} edges "
        f"(underlying edges represented = {total_edges:,})."
     )
    draw_attack_graph(
        subnet_graph,
        subnet_role_map,
        title='Label-agnostic attack flow (subnet /24)',
        layout='spring',
        seed=28,
        figsize=(18, 14),
        node_size=135,
        edge_alpha=0.25,
        edge_weight_attr='edge_count',
        attack_edge_width=2.6,
        benign_edge_width=1.2,
        save_path=run_dir / f'label_agnostic_{chain_prefix}_subnet_graph.png',
     )

    print('\nSaved network visualization artifacts:')
    for artifact in sorted(run_dir.glob('*_graph.png')):
        print(f'  {artifact.name}')

print(f'\n{"="*80}')
print('✓ VISUALIZATION COMPLETE')
print(f'{"="*80}')
print(f'Multi-hop chain CSV files generated by CART analyzer:')
print(f'  - label_aware_{N_HOPS}hop_chains.csv')
print(f'  - label_agnostic_{N_HOPS}hop_chains.csv')
print(f'Network graphs saved to: {run_dir}')
print(f'{"="*80}')

In [ ]:
# Optional: stop the container when finished
controller.close()
if SHUTDOWN_AFTER_RUN:
    controller.stop()
    print('Neo4j container stopped.')
else:
    print('Neo4j container left running; call controller.stop() if you want to shut it down.')

In [ ]:
# Generate all thesis artifacts (tables, figures, summaries)
if GENERATE_THESIS_ARTIFACTS:
    print('\n' + '=' * 80)
    print('GENERATING THESIS ARTIFACTS')
    print('=' * 80)
    import subprocess
    import sys
    
    result = subprocess.run(
        [sys.executable, 'generate_all_thesis_artifacts.py'],
        cwd=str(Path.cwd()),
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    if result.returncode == 0:
        print('\n✓ Thesis artifacts generated successfully!')
        print('  Check thesis_figures/ for all outputs')
    else:
        print(f'\n⚠ Artifact generation completed with code {result.returncode}')
else:
    print('\nSkipping artifact generation (set GENERATE_THESIS_ARTIFACTS=True to enable)')